# A7 — Painéis Espaciais e Dinâmica Territorial

Este notebook demonstra a aplicação longitudinal do curso Espaçoso,
organizando dados unidade–tempo e comparando especificações não espaciais
e espaciais.

## Estrutura
1. Geração de dados simulados
2. Construção e validação do painel
3. Tratamento de lacunas e painel desbalanceado
4. Efeitos fixos de unidade e temporais
5. Modelo de lag espacial (IV/2SLS)
6. Modelo de erro espacial (GM)
7. Modelo dinâmico e limites de identificação
8. Comparação de resultados e diagnóstico


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import pandas as pd

from paineis_espaciais.panel import build_panel, lag_column
from paineis_espaciais.weights import build_weights, WeightSpec, matrix_diagnostics
from paineis_espaciais.models import fit_fe, fit_spatial_lag, fit_spatial_error, compare_models


## 1. Geração de dados simulados

Usamos o script `scripts/create_demo_data.py` para gerar:
- `painel_demo.csv`: painel balanceado com N=20 unidades e T=5 períodos
- `painel_demo_desbalanceado.csv`: versão com 3 observações removidas
- `pesos_queen.csv`: matriz de contiguidade rainha row-standardizada


In [ ]:
import subprocess
subprocess.run(['python', '../scripts/create_demo_data.py', '--output-dir', '../data/demo'], check=True)
print('Dados gerados com sucesso.')


## 2. Construção e validação do painel

In [ ]:
df = pd.read_csv('../data/demo/painel_demo.csv')
print(f'Shape: {df.shape}')
df.head()


In [ ]:
panel = build_panel(df, unit_col='unit_id', time_col='time_id')
print(f'Unidades: {panel.n_units}')
print(f'Períodos: {panel.n_periods}')
print(f'Balanceado: {panel.balanced}')
print(f'Lacunas: {panel.missing_cells}')


## 3. Painel desbalanceado e tratamento de lacunas

In [ ]:
df_unbal = pd.read_csv('../data/demo/painel_demo_desbalanceado.csv')
panel_unbal = build_panel(df_unbal, unit_col='unit_id', time_col='time_id')
print(f'Painel desbalanceado — lacunas: {panel_unbal.missing_cells}')

panel_filled = build_panel(
    df_unbal,
    unit_col='unit_id',
    time_col='time_id',
    gap_strategy='forward_fill',
    gap_limit=1,
)
print(f'Após preenchimento — lacunas: {panel_filled.missing_cells}')
print('Nota: o preenchimento é feito DENTRO de cada unidade, sem contaminação entre unidades.')


## 4. Efeitos fixos de unidade e temporais

In [ ]:
fe_unit = fit_fe(panel, target='y', predictors=['x1', 'x2'], fixed_effects='unit', spec_name='FE-Unidade')
fe_time = fit_fe(panel, target='y', predictors=['x1', 'x2'], fixed_effects='time', spec_name='FE-Tempo')
fe_2way = fit_fe(panel, target='y', predictors=['x1', 'x2'], fixed_effects='two_way', spec_name='FE-Bidirecional')

for fe in [fe_unit, fe_time, fe_2way]:
    print(f'{fe.spec_name}: R² = {fe.result.rsquared:.4f}, n = {fe.n_obs}')


### Interpretação dos efeitos fixos

- **Efeitos fixos de unidade**: eliminam toda variação between-unit, controlando para
  características não observadas constantes de cada unidade (cultura, geografia, etc.).
- **Efeitos fixos de tempo**: eliminam choques macroeconômicos comuns a todas as unidades
  em cada período.
- **Bidirecional**: combina ambos, maximizando o controle mas reduzindo os graus de liberdade.


## 5. Matriz de pesos espaciais

In [ ]:
wspec = WeightSpec(
    name='queen',
    path=Path('../data/demo/pesos_queen.csv'),
    transformation='row_standardized',
)
ids = panel.data.index.get_level_values(0).unique().tolist()
W = build_weights(wspec, ids)

diag = matrix_diagnostics(W, name='queen')
pd.Series(diag)


### Justificativa da matriz constante

A matriz W é definida pela contiguidade geográfica entre regiões, que é **constante no tempo**
(fronteiras não mudam ao longo do período de análise). Isso simplifica a estimação e é
consistente com a literatura de painéis espaciais. Uma matriz variável no tempo seria
necessária se as relações entre unidades (ex.: comércio, migração) variassem substantivamente.


## 6. Modelo de lag espacial (IV/2SLS)

In [ ]:
sl = fit_spatial_lag(
    panel, W=W, target='y', predictors=['x1', 'x2'],
    fixed_effects='unit', spec_name='Lag-Espacial',
)
print(f'ρ (rho) = {sl.rho_or_lambda:.4f}')
print(f'R² = {sl.r_squared:.4f}')
print(f'\nNota de identificação:\n{sl.identification_note}')


## 7. Modelo de erro espacial (GM)

In [ ]:
se = fit_spatial_error(
    panel, W=W, target='y', predictors=['x1', 'x2'],
    fixed_effects='unit', spec_name='Erro-Espacial',
)
print(f'λ (lambda) = {se.rho_or_lambda:.4f}')
print(f'R² = {se.r_squared:.4f}')
print(f'\nNota de identificação:\n{se.identification_note}')


## 8. Modelo dinâmico e limites de identificação

In [ ]:
import warnings

panel_lagged = build_panel(df, unit_col='unit_id', time_col='time_id')
panel_lagged_data = lag_column(panel_lagged.data, 'y', n_lags=1)
from paineis_espaciais.panel import PanelData
panel_with_lag = PanelData(
    data=panel_lagged_data,
    unit_col=panel_lagged.unit_col,
    time_col=panel_lagged.time_col,
    balanced=panel_lagged.balanced,
    n_units=panel_lagged.n_units,
    n_periods=panel_lagged.n_periods,
    missing_cells=panel_lagged.missing_cells,
)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    sl_dyn = fit_spatial_lag(
        panel_with_lag, W=W, target='y', predictors=['x1', 'x2'],
        fixed_effects='unit', spec_name='Lag-Dinâmico', dynamic=True,
    )
    for warning in w:
        print(f'AVISO: {warning.message}')

print(f'ρ = {sl_dyn.rho_or_lambda:.4f}, dinâmico = {sl_dyn.dynamic}')


## 9. Comparação de resultados e diagnóstico

In [ ]:
comparison = compare_models(fe_unit, [sl, se, sl_dyn])
comparison


## 10. Execução via CLI

O pipeline completo pode ser executado via linha de comando:

```bash
# Gerar dados de demonstração
python scripts/create_demo_data.py --output-dir data/demo

# Executar pipeline
paineis-espaciais config/demo.yml
```
